# 03 Production SQL Analytics Layer

## Portfolio Project: Customer Retention & Revenue Risk Analytics

### Overview
This notebook initializes an in-memory SQLite database, loads the cleaned dataset alongside model predictions, and executes 10 production SQL queries utilizing CTEs, Window Functions (`RANK()`, `DENSE_RANK()`, `SUM() OVER()`), CASE statements, and aggregations.

---


In [ ]:
import sys, os, sqlite3
import pandas as pd

sys.path.append(r'c:\Users\Atul\Desktop\project-2\customer-retention-revenue-risk-analytics')

proc_path = r'c:\Users\Atul\Desktop\project-2\customer-retention-revenue-risk-analytics\data\processed\cleaned_telco_churn.csv'
risk_path = r'c:\Users\Atul\Desktop\project-2\customer-retention-revenue-risk-analytics\data\processed\churn_predictions_risk_matrix.csv'

df_proc = pd.read_csv(proc_path)
df_risk = pd.read_csv(risk_path)

# Connect to in-memory SQLite database
conn = sqlite3.connect(':memory:')

# Write dataframes to SQLite tables
df_proc.to_sql('customers', conn, index=False, if_exists='replace')
df_risk.to_sql('churn_predictions', conn, index=False, if_exists='replace')

print("SQLite Database initialized with tables: 'customers' and 'churn_predictions'")


### SQL Query 1: Overall Churn Rate & Baseline KPIs


In [ ]:
q1 = """
SELECT 
    COUNT(CustomerID) AS TotalCustomers,
    SUM(CASE WHEN Churn_Numeric = 0 THEN 1 ELSE 0 END) AS ActiveCustomers,
    SUM(CASE WHEN Churn_Numeric = 1 THEN 1 ELSE 0 END) AS ChurnedCustomers,
    ROUND(100.0 * SUM(CASE WHEN Churn_Numeric = 1 THEN 1 ELSE 0 END) / COUNT(CustomerID), 2) AS ChurnRatePct,
    ROUND(AVG(MonthlyCharges), 2) AS AvgMonthlyCharges,
    ROUND(SUM(MonthlyCharges), 2) AS TotalMonthlyMRR,
    ROUND(SUM(CASE WHEN Churn_Numeric = 1 THEN MonthlyCharges ELSE 0 END), 2) AS HistoricalMonthlyRevenueLost
FROM customers;
"""
pd.read_sql_query(q1, conn)


### SQL Query 6: Rank Customer Segments by Revenue Exposure using `RANK()` Window Function


In [ ]:
q6 = """
WITH SegmentExposure AS (
    SELECT 
        Contract,
        PaymentMethod,
        InternetService,
        COUNT(CustomerID) AS SegmentSize,
        SUM(CASE WHEN Churn_Numeric = 1 THEN 1 ELSE 0 END) AS ChurnedCount,
        ROUND(SUM(CASE WHEN Churn_Numeric = 1 THEN MonthlyCharges ELSE 0 END), 2) AS MonthlyRevenueLost
    FROM customers
    GROUP BY Contract, PaymentMethod, InternetService
)
SELECT 
    Contract,
    PaymentMethod,
    InternetService,
    SegmentSize,
    ChurnedCount,
    MonthlyRevenueLost,
    RANK() OVER (ORDER BY MonthlyRevenueLost DESC) AS ExposureRank
FROM SegmentExposure
ORDER BY ExposureRank ASC
LIMIT 10;
"""
pd.read_sql_query(q6, conn)


### SQL Query 10: Cumulative Revenue Exposure Window Function (`SUM() OVER (...)`)


In [ ]:
q10 = """
WITH RankedActiveRisk AS (
    SELECT 
        CustomerID,
        Contract,
        MonthlyCharges,
        ChurnProbability,
        ExpectedRevenueAtRisk,
        PriorityScore
    FROM churn_predictions
    WHERE Churn_Numeric = 0
)
SELECT 
    CustomerID,
    Contract,
    MonthlyCharges,
    ROUND(ChurnProbability, 4) AS ChurnProbability,
    ROUND(ExpectedRevenueAtRisk, 2) AS ExpectedRevenueAtRisk,
    ROUND(SUM(ExpectedRevenueAtRisk) OVER (ORDER BY PriorityScore DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2) AS CumulativeRevenueRisk
FROM RankedActiveRisk
ORDER BY PriorityScore DESC
LIMIT 15;
"""
pd.read_sql_query(q10, conn)
